# Mixture rules

In [2]:
import pandas as pd
import sqlite3

## Step 1. Extract info from DB

### Extraction from DB
    extracts from the DB:
        1. SLC
        2. Toxicity
        3. Corrosion/irritation
        4. Sensitization
        5. Aquatic toxicity (acute and chronic)


In [4]:
### Extract LD50 or LC50 for each CAS that I give in a list for CAS in DB (oral, dermal, inhalation)


cas_list = ['10-00-1', '10-00-0', '110-54-3', '1592-23-0']
db_path = '/Users/juliakulpa/Desktop/DB_tests_mixture_rules/C2Cdatabase.db'

def extract_info_from_DB(cas_list, db_path):
    ''' The output is a df with cols: CAS and each CPL/tox info, for each CAS in the row
    Works on:
    database: maindb="C2C_DATABASE"
    cas is always in the col named "ref"
    extracts from the DB:
        1. SLC
        2. Toxicity
        3. Corrosion/irritation
        4. Sensitization
        5. Aquatic toxicity (acute and chronic)
     SCl is saved as a separate df
     Other info is saved in a separate df
     at the end: both df are merged on outer (keeping all values)
     '''
    # Connect to the Db and establish cursor
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Store results
    results = []
    SCL_results = []

    # Gen information which cols are in SCL:
    table_name = 'SCONCLIM'
    exclude_cols = {"ID", "ref"}
    cursor.execute(f"PRAGMA table_info({table_name})")
    cols = [row[1] for row in cursor.fetchall()]
    selected_cols_SCL = [col for col in cols if col not in exclude_cols]
    selected_cols_SCL_sql = ", ".join([f'"{col}"' for col in selected_cols_SCL])

    # Loop over each CAS in the provided list
    for cas in cas_list:
        ### 1. SCL ####
        # SCL:
        query = f'''
        SELECT {selected_cols_SCL_sql}
        FROM "{table_name}"
        WHERE "ref" = ?
        '''

        cursor.execute(query, (cas,))
        rows = cursor.fetchall()
        df_SCL = pd.DataFrame(rows, columns=selected_cols_SCL)
        df_SCL.insert(0, "CAS", cas)
        SCL_results.append(df_SCL)

        ### 2. Toxicity ####
        # ORALTOX table for LD50_oral
        cursor.execute(f"""
            SELECT
                "Oral Acute: LD50 =",
                "Oral toxicity Acute Tox classified"
            FROM ORALTOX WHERE ref = ?
        """, (cas,))
        oral_data = cursor.fetchone()
        oral_ld50 = oral_data[0] if oral_data else None
        oral_CLP_class = oral_data[1] if oral_data else None

        # INHALTOX table for LC50 (gas, vapour, dust/mist/aerosol)
        cursor.execute(f"""
            SELECT
                "Inhalative toxicity Acute: LC50 (gas) =",
                "Inhalative toxicity Acute: LC50 (vapor) =",
                "Inhalative toxicity Acute: LC50 (dust/mist/aerosol) =",
                "Inhalative toxicity Acute Tox classification"
            FROM INHALTOX WHERE ref = ?
        """, (cas,))
        inhalation_data = cursor.fetchone()
        lc50_gas = inhalation_data[0] if inhalation_data else None
        lc50_vapour = inhalation_data[1] if inhalation_data else None
        lc50_dust_mist_aerosol = inhalation_data[2] if inhalation_data else None
        inhal_CLP_class = inhalation_data[3] if inhalation_data else None

        # DERMALTOX table for LD50_dermal
        cursor.execute(f"""
            SELECT
                "Dermal Acute: LD50 =",
                "Dermal toxicity Acute Tox classified"
            FROM DERMALTOX WHERE ref = ?
        """, (cas,))
        dermal_data = cursor.fetchone()
        dermal_ld50 = dermal_data[0] if dermal_data else None
        dermal_CLP_class = dermal_data[1] if dermal_data else None

        ### 3. CORR / IRR ###
        cursor.execute(f"""
            SELECT
                "Skin irritation classification",
                "Eye irritation classification",
                "Respiratory irritation classification"
            FROM IRRITCOR WHERE ref = ?
        """, (cas,))
        irritation_data = cursor.fetchone()
        skin_irr = irritation_data[0] if irritation_data else None
        eye_irr = irritation_data[1] if irritation_data else None
        reps_irr = irritation_data[2] if irritation_data else None

        ### 4. SENSITISATION ###
        cursor.execute(f"""
            SELECT
                "Skin sensitization CLP classification",
                "Respiratory sensitization CLP classification"
            FROM SENSITISATION WHERE ref = ?
        """, (cas,))
        sensitisation_data = cursor.fetchone()
        skin_sensitisation = sensitisation_data[0] if sensitisation_data else None
        resp_sensitisation = sensitisation_data[1] if sensitisation_data else None

        ### 5. AQUATIC TOX ###

        # M factor, Aquatic tox acute & chronic
        cursor.execute(f"""
            SELECT
                "Aquatic toxicity Acute Tox classified",
                "Aquatic toxicity Chronic Tox classified",
                "M factor"
            FROM AQUATOX WHERE ref = ?
        """, (cas,))
        aquatic_tox_data = cursor.fetchone()
        aquatic_tox_acute = aquatic_tox_data[0] if aquatic_tox_data else None
        aquatic_tox_chronic = aquatic_tox_data[1] if aquatic_tox_data else None
        m_factor = aquatic_tox_data[2] if aquatic_tox_data else None

        # Fish toxicity
        cursor.execute(f"""
            SELECT
                "Fish toxicity Acute: LC50 (96h) =",
                "Fish toxicity Chronic: NOEC =",
                "Fish toxicity Acute QSAR: LC50 =",
                "Fish toxicity Chronic QSAR: NOEC ="
            FROM FISHTOX WHERE ref = ?
        """, (cas,))
        fish_tox_data = cursor.fetchone()
        fish_lc50 = fish_tox_data[0] if fish_tox_data else None
        fish_noec = fish_tox_data[1] if fish_tox_data else None
        fish_lc50_qsar = fish_tox_data[2] if fish_tox_data else None
        fish_noec_qsar = fish_tox_data[3] if fish_tox_data else None

        # Daphnae / invertebrate toxicity
        cursor.execute(f"""
            SELECT
                "Invertebrate toxicity Acute: L(E)C50 (48h) =",
                "Invertebrae toxicity Chronic: NOEC =",
                "Invertebrae toxicity Acute QSAR: LC50 =",
                "Invertebrae toxicity Chronic QSAR: NOEC ="
            FROM INVTOX WHERE ref = ?
        """, (cas,))
        daph_tox_data = cursor.fetchone()
        daph_lc50 = daph_tox_data[0] if daph_tox_data else None
        daph_noec = daph_tox_data[1] if daph_tox_data else None
        daph_lc50_qsar = daph_tox_data[2] if daph_tox_data else None
        daph_noec_qsar = daph_tox_data[3] if daph_tox_data else None

        # Algae toxicity
        cursor.execute(f"""
            SELECT
                "Algae toxicity Acute: L(E)C50 (72/96h) =",
                "Algae toxicity Chronic: NOEC =",
                "Algae toxicity Acute QSAR: LC50 =",
                "Algae toxicity Chronic QSAR: NOEC ="
            FROM ALGAETOX WHERE ref = ?
        """, (cas,))
        algae_tox_data = cursor.fetchone()
        algae_lc50 = algae_tox_data[0] if algae_tox_data else None
        algae_noec = algae_tox_data[1] if algae_tox_data else None
        algae_lc50_qsar = algae_tox_data[2] if algae_tox_data else None
        algae_noec_qsar = algae_tox_data[3] if algae_tox_data else None

        ### Append results for each CAS:
        # Add the data for the current CAS to the results list
        results.append({
            "CAS": cas,
            "LD50_oral": oral_ld50,
            "LC50_gas": lc50_gas,
            "LC50_vapour": lc50_vapour,
            "LC50_dust_mist_aerosol": lc50_dust_mist_aerosol,
            "LD50_dermal": dermal_ld50,
            "CLP oral class": oral_CLP_class,
            "CLP dermal class": dermal_CLP_class,
            "CLP inhalation class": inhal_CLP_class,
            "skin_corr_irr": skin_irr,
            "eye_corr_irr": eye_irr,
            "reps_corr_irr": reps_irr,
            "skin_sensitisation": skin_sensitisation,
            "resp_sensitisation": resp_sensitisation,
            "aquatic_tox_acute": aquatic_tox_acute,
            "aquatic_tox_chronic": aquatic_tox_chronic,
            "m_factor": m_factor,
            "fish_lc50": fish_lc50,
            "fish_noec": fish_noec,
            "fish_lc50_qsar": fish_lc50_qsar,
            "fish_noec_qsar": fish_noec_qsar,
            "daph_lc50": daph_lc50,
            "daph_noec": daph_noec,
            "daph_lc50_qsar": daph_lc50_qsar,
            "daph_noec_qsar": daph_noec_qsar,
            "algae_lc50": algae_lc50,
            "algae_noec": algae_noec,
            "algae_lc50_qsar": algae_lc50_qsar,
            "algae_noec_qsar": algae_noec_qsar
        })

    # obtain results for SCL
    df_final_SCL = pd.concat(SCL_results, ignore_index=True)

    # Convert the results into a df
    df_info = pd.DataFrame(results)

    # connect both df
    df = df_info.merge(df_final_SCL, on="CAS", how="outer")

    # Close the database connection
    conn.close()

    # Return the DataFrame
    return df
df_toxicity = extract_info_from_DB(cas_list, db_path)
CAS_in_tox = df_toxicity["CAS"].unique().tolist()
print(df_toxicity)
df_toxicity.to_excel("/Users/juliakulpa/Desktop/DB_tests_mixture_rules/toxicity.xlsx")

         CAS LD50_oral LC50_gas LC50_vapour LC50_dust_mist_aerosol  \
0    10-00-0       640     4234         463                   3232   
1    10-00-1       640     4234         463                   3232   
2   110-54-3       NaN      NaN         NaN                    NaN   
3  1592-23-0      2000      NaN         NaN                    NaN   

  LD50_dermal                                     CLP oral class  \
0         213           Acute Tox. 4: H302: Harmful if swallowed   
1         213           Acute Tox. 4: H302: Harmful if swallowed   
2        3500  Asp. Tox. 1: H304: May be fatal if swallowed a...   
3        2000                                                NaN   

  CLP dermal class                  CLP inhalation class  \
0   Not classified  Acute Tox. 2: H330: Fatal if inhaled   
1   Not classified  Acute Tox. 2: H330: Fatal if inhaled   
2              NaN                                   NaN   
3              NaN                                   NaN   

       

## Step 2: Calculations

### Toxicity
Calculations of ATE for the mixture
Calculate ATE for each homogenous material calculate (all relevant CAS in the homogenous mixture)


In [9]:
# get the file with CAS & corresponding percentage
df_product = pd.read_excel('/Users/juliakulpa/Desktop/DB_tests_mixture_rules/Test_for_mixture_rules_v2.xlsx')

def calculate_ATE(df_calculation,  hom_materials, ate_specification):
    """
    :param df_calculation: dataframe with CAS and their associated toxicity
    :param ate_specification: we should specify which ATE to calculate corresponding to LD50 in the df_calcuation e.g. if we want to calculate oral one we should say LD50_oral
    :param hom_materials: homogenous materials we should calculate ATE for
    :return: dataftrame with ATE for each homogenous material
    """
    # LD50 for oral/inhalation/dermal etc.
    LD50 = ate_specification

    df_calculation = df_calculation.copy()

    # save the highest value of contribution of hom mat
    df_calculation["conc_hom_mat"] = df_calculation[["min_contribution_hom_mat", "max_contribution_hom_mat"]].max(axis=1)
    # force the data to be numeric
    df_calculation[LD50] = pd.to_numeric(df_calculation[LD50], errors='coerce')

    ### Step 1. check the concentration and exclude the one below 0.1 %
    df_calculation = df_calculation[df_calculation["conc_hom_mat"]>= 0.001].copy()

    ### Step 2. exclude the values that are below 1 % and are classified as cat.4 ( np. LD50 oral > 2000)
    if LD50 == "LD50_oral":
        exclusion_value = 2000
        CLP_info = "CLP oral class"
        LD50_tox_1 = 0.5
        LD50_tox_2 = 5
        LD50_tox_3 = 100
        LD50_tox_4 = 500
    if LD50 == "LD50_dermal":
        exclusion_value = 2000
        CLP_info = "CLP dermal class"
        LD50_tox_1 = 5
        LD50_tox_2 = 50
        LD50_tox_3 = 300
        LD50_tox_4 = 1100
    if LD50 == "LC50_gas":
        exclusion_value = 20000
        CLP_info = "CLP inhalation class"
        LD50_tox_1 = 10
        LD50_tox_2 = 100
        LD50_tox_3 = 700
        LD50_tox_4 = 4500
    if LD50 == "LC50_vapour":
        exclusion_value = 20
        CLP_info = "CLP inhalation class"
        LD50_tox_1 = 0.05
        LD50_tox_2 = 0.5
        LD50_tox_3 = 3
        LD50_tox_4 = 5
    if LD50 == "LC50_dust_mist_aerosol":
        exclusion_value = 5
        CLP_info = "CLP inhalation class"
        LD50_tox_1 = 0.005
        LD50_tox_2 = 0.05
        LD50_tox_3 = 0.5
        LD50_tox_4 = 1.5

    df_calculation = df_calculation.loc[~((df_calculation["conc_hom_mat"] < 0.01) & (df_calculation[LD50] > exclusion_value))].copy()

    # add values of LD50 for chemicals that are classified in a category but do not have an LD50 value
    # tox cat 1
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 1", na=False, regex=False),LD50] = LD50_tox_1
    # tox cat 2
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 2", na=False, regex=False),LD50] = LD50_tox_2
    # tox cat 3
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 3", na=False, regex=False),LD50] = LD50_tox_3
    # tox cat 4
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 4", na=False, regex=False),LD50] = LD50_tox_4
    ### Step 3. Calculate ATE based on:

    # 100 – (∑n % chemicals unknown > 10%) / ATE mixture = ∑n % chemical is in formulation/LD50 or LC50
    # iterate for each hom mat:
    ATE_for_each_material = []

    for hom_material in hom_materials:
        df_calc_hom_material = df_calculation.loc[df_product["Homogenous Material"] == hom_material]

        # Sum of chemical with unknown LD50 that are not "not classified" in CLP

        condition_to_be_unknown_chemical = (df_calc_hom_material["conc_hom_mat"] > 0.1 ) & (df_calc_hom_material[LD50].isna() & (df_calc_hom_material[CLP_info] != "Not classified"))
        unknown_LD_class_chemicals = df_calc_hom_material.loc[condition_to_be_unknown_chemical, "CAS"].tolist()
        sum_unknown_chemicals = df_calc_hom_material.loc[condition_to_be_unknown_chemical, "conc_hom_mat"].sum()

        # calculate 100 – (∑n % chemicals unknown > 10%)
        adjusted_100 = 100 - sum_unknown_chemicals

        # n % chemical is in formulation/LD50 (PER CHEMICAL)
        df_calc_hom_material["conc_divided_by_LD50"] = df_calc_hom_material["conc_hom_mat"]/df_calculation[LD50]

        # ∑n % chemical is in formulation/LD50 (SUM ALL)
        sum_constituents = df_calc_hom_material["conc_divided_by_LD50"].sum()

        # ATE = 100 – (∑n % chemicals unknown > 10%) / ∑n % chemical is in formulation/LD50 or LC50
        ate = adjusted_100 / sum_constituents

        # ATE_for_each_material[hom_material] = round(float(ate),2)

        ATE_for_each_material.append({
                "hom_material": hom_material,
                f"ATE_based_on_{LD50}": round(float(ate),2)})

    df = pd.DataFrame(ATE_for_each_material)
    return df, unknown_LD_class_chemicals
def obtain_all_ATE_df(df_calculation, hom_materials, all_ld_50_or_lc_50_options):
    ''''
    :param df_calculation: dataframe with CAS and their associated toxicity
    :param hom_materials: homogenous materials we should calculate ATE for
    :param all_ld_50_or_lc_50_options (specify which for which LD50 (oral, demral etc.) to calculate ATE)
    :return: dataftrame with ATE (oral, dermal etc.) for each homogenous material, list of chemicals with unknown ATE but not "not classified"
     '''
    final_df = None
    for ld_50_or_lc_50 in all_ld_50_or_lc_50_options:
        df, unknown_LD_class_chemicals = calculate_ATE(df_calculation, hom_materials, ate_specification = ld_50_or_lc_50)
        if final_df is None:
            final_df = df
        else:
            final_df = final_df.merge(df, on="hom_material", how="outer")
    return final_df, unknown_LD_class_chemicals

# get the unique hom materials
hom_materials = df_product["Homogenous Material"].unique().tolist()
# merge the df
df_calculation = pd.merge(df_product, df_toxicity, on="CAS", how="left")
all_ld_50_or_lc_50_options = ["LD50_oral","LC50_gas","LC50_vapour","LC50_dust_mist_aerosol","LD50_dermal"]
final_df, unknown_LD_class_chemicals = obtain_all_ATE_df(df_calculation, hom_materials, all_ld_50_or_lc_50_options)
print(final_df)
print("Unkonwn chemicals:",unknown_LD_class_chemicals)

  hom_material  ATE_based_on_LD50_oral  ATE_based_on_LC50_gas  \
0            A                   71.40             1051090.50   
1            B                72562.36              498117.65   

   ATE_based_on_LC50_vapour  ATE_based_on_LC50_dust_mist_aerosol  \
0                 114939.75                            802344.00   
1                  54470.59                            380235.29   

   ATE_based_on_LD50_dermal  
0                  48124.72  
1                  24748.74  
Unkonwn chemicals: []


Classifing the mixture based on calculated ATE

In [17]:
def classify_mixture_CLP(df_with_ate):
    df_with_ate = df_with_ate.copy()

    # assess oral tox
    if 'ATE_based_on_LD50_oral' in df_with_ate.columns:
        tox_category = "Acute toxicity oral"
        ate_value = 'ATE_based_on_LD50_oral'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 5, tox_category] = "Cat. 1"
        df_with_ate.loc[df_with_ate[ate_value].between(5, 50, inclusive="right"), tox_category] = "Cat. 2"
        df_with_ate.loc[df_with_ate[ate_value].between(50, 300, inclusive="right"), tox_category] = "Cat. 3"
        df_with_ate.loc[df_with_ate[ate_value].between(300, 2000, inclusive="right"), tox_category] = "Cat. 4"

    # assess dermal tox
    if 'ATE_based_on_LD50_dermal' in df_with_ate.columns:
        tox_category = "Acute toxicity dermal"
        ate_value = 'ATE_based_on_LD50_dermal'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 50, tox_category] = "Cat. 1"
        df_with_ate.loc[df_with_ate[ate_value].between(50, 200, inclusive="right"), tox_category] = "Cat. 2"
        df_with_ate.loc[df_with_ate[ate_value].between(200, 1000, inclusive="right"), tox_category] = "Cat. 3"
        df_with_ate.loc[df_with_ate[ate_value].between(1000, 2000, inclusive="right"), tox_category] = "Cat. 4"

    # assess inhal tox gases
    if 'ATE_based_on_LC50_gas' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (gases)"
        ate_value = 'ATE_based_on_LC50_gas'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 100, tox_category] = "Cat. 1"
        df_with_ate.loc[df_with_ate[ate_value].between(100, 500, inclusive="right"), tox_category] = "Cat. 2"
        df_with_ate.loc[df_with_ate[ate_value].between(500, 2500, inclusive="right"), tox_category] = "Cat. 3"
        df_with_ate.loc[df_with_ate[ate_value].between(2500, 20000, inclusive="right"), tox_category] = "Cat. 4"

    # assess inhal tox vapour
    if 'ATE_based_on_LC50_vapour' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (vapour)"
        ate_value = 'ATE_based_on_LC50_vapour'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 0.5, tox_category] = "Cat. 1"
        df_with_ate.loc[df_with_ate[ate_value].between(0.5, 2, inclusive="right"), tox_category] = "Cat. 2"
        df_with_ate.loc[df_with_ate[ate_value].between(2, 10, inclusive="right"), tox_category] = "Cat. 3"
        df_with_ate.loc[df_with_ate[ate_value].between(10, 20, inclusive="right"), tox_category] = "Cat. 4"

    # assess inhal tox dust/mist
    if 'ATE_based_on_LC50_dust_mist_aerosol' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (dust/mist)"
        ate_value = 'ATE_based_on_LC50_dust_mist_aerosol'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 0.05, tox_category] = "Cat. 1"
        df_with_ate.loc[df_with_ate[ate_value].between(0.05, 0.5, inclusive="right"), tox_category] = "Cat. 2"
        df_with_ate.loc[df_with_ate[ate_value].between(0.5, 1, inclusive="right"), tox_category] = "Cat. 3"
        df_with_ate.loc[df_with_ate[ate_value].between(1, 5, inclusive="right"), tox_category] = "Cat. 4"

    return df_with_ate


assessment for C2C

In [18]:
def classify_mixture_C2C(df_with_ate):
    df_with_ate = df_with_ate.copy()

    # assess oral tox
    if 'ATE_based_on_LD50_oral' in df_with_ate.columns:
        tox_category = "Acute toxicity oral C2C"
        ate_value = 'ATE_based_on_LD50_oral'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 300, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(300, 2000, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>2000, tox_category] = "GREEN"


    # assess dermal tox
    if 'ATE_based_on_LD50_dermal' in df_with_ate.columns:
        tox_category = "Acute toxicity dermal C2C"
        ate_value = 'ATE_based_on_LD50_dermal'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 1000, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(1000, 2000, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>2000, tox_category] = "GREEN"

    # assess inhal tox gases
    if 'ATE_based_on_LC50_gas' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (gases) C2C"
        ate_value = 'ATE_based_on_LC50_gas'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 10, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(10, 20, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>20, tox_category] = "GREEN"

    # assess inhal tox vapour
    if 'ATE_based_on_LC50_vapour' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (vapour) C2C"
        ate_value = 'ATE_based_on_LC50_vapour'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 10, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(10, 20, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>20, tox_category] = "GREEN"

    # assess inhal tox dust/mist
    if 'ATE_based_on_LC50_dust_mist_aerosol' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (dust/mist) C2C"
        ate_value = 'ATE_based_on_LC50_dust_mist_aerosol'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 1, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(1, 5, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>5, tox_category] = "GREEN"

    # The assessment of the whole mixture
    df_with_ate["C2C acute toxicity"] = None

    # fill in red first
    df_with_ate.loc[(df_with_ate["Acute toxicity oral C2C"] == "RED") | (df_with_ate["Acute toxicity dermal C2C"] == "RED") |
                    (df_with_ate["Acute toxicity inhalation (gases) C2C"] == "RED") | (df_with_ate["Acute toxicity inhalation (vapour) C2C"] == "RED")
                    | (df_with_ate["Acute toxicity inhalation (dust/mist) C2C"] == "RED"), "C2C acute toxicity"] = "RED"


    return df_with_ate

### tests:

In [ ]:
### Toxicity:

# get the file with CAS & corresponding percentage
df_product = pd.read_excel('/Users/juliakulpa/Desktop/DB_tests_mixture_rules/Test_for_mixture_rules.xlsx')

# get the unique hom materials
hom_materials = df_product["Homogenous Material"].unique().tolist()

def ate_oral(hom_materials, df_product, df_toxicity):
    """
    :param hom_materials: df with homogenous materials
    :param df_product: df with a product
    :param df_toxicity: df with hazards
    :return: dictionary with oral ATE
    """
    # calculate ATE for each hom material
    oral_ate_dict = {}
    # loop over hom materials
    for hom_material in hom_materials:
        # create a df per hom material
        df_hom_material = df_product.loc[df_product["Homogenous Material"] == hom_material]

        df_calculation = pd.merge(df_hom_material, df_toxicity, on="CAS", how="left")

        # Calculate ATE oral for max_contribution_hom_mat
        LD50_oral = 'LD50_oral'
        max_percent_in_hom_mat = "max_contribution_hom_mat"

        #make sure the cols are numeric:
        df_calculation[LD50_oral] = pd.to_numeric(df_calculation[LD50_oral ], errors='coerce')
        df_calculation[max_percent_in_hom_mat] = pd.to_numeric(df_calculation[max_percent_in_hom_mat], errors='coerce')

        # drop all values with NaN
        df_calculation = df_calculation.dropna(subset=[LD50_oral, max_percent_in_hom_mat])

        # calcuale in a table form conc / LD50 for each component
        df_calculation["conc_divided_by_LD50"] = df_calculation[max_percent_in_hom_mat]/df_calculation[LD50_oral]

        # sum conc / LD50 of the components
        ate_100 = df_calculation["conc_divided_by_LD50"].sum()

        # calculate the ATE 100/ATE = sum(conc / LD50) -> ATE = 100/sum(conc / LD50)

        ate_oral = 100 / ate_100

        oral_ate_dict[hom_material] = float(ate_oral)

    return oral_ate_dict

def ate_inhalation_gas(hom_materials, df_product, df_toxicity):
    """
    :param hom_materials: df with homogenous materials
    :param df_product: df with a product
    :param df_toxicity: df with hazards
    :return: dictionary with inhalation_gas ATE
    """
    # calculate ATE for each hom material
    inhalation_gas_ate_dict = {}
    # loop over hom materials
    for hom_material in hom_materials:
        # create a df per hom material
        df_hom_material = df_product.loc[df_product["Homogenous Material"] == hom_material]

        df_calculation = pd.merge(df_hom_material, df_toxicity, on="CAS", how="left")

        # Calculate ATE oral for max_contribution_hom_mat
        LC50 = 'LC50_gas'
        max_percent_in_hom_mat = "max_contribution_hom_mat"

        #make sure the cols are numeric:
        df_calculation[LC50] = pd.to_numeric(df_calculation[LC50], errors='coerce')
        df_calculation[max_percent_in_hom_mat] = pd.to_numeric(df_calculation[max_percent_in_hom_mat], errors='coerce')

        # drop all values with NaN
        df_calculation = df_calculation.dropna(subset=[LC50, max_percent_in_hom_mat])

        # calcuale in a table form conc / LD50 for each component
        df_calculation["conc_divided_by_LD50"] = df_calculation[max_percent_in_hom_mat]/df_calculation[LC50]

        # sum conc / LD50 of the components
        ate_100 = df_calculation["conc_divided_by_LD50"].sum()

        # calculate the ATE 100/ATE = sum(conc / LD50) -> ATE = 100/sum(conc / LD50)

        ate = 100 / ate_100

        inhalation_gas_ate_dict[hom_material] = float(ate)

    return inhalation_gas_ate_dict

def ate_inhalation_vapour(hom_materials, df_product, df_toxicity):
    """
    :param hom_materials: df with homogenous materials
    :param df_product: df with a product
    :param df_toxicity: df with hazards
    :return: dictionary with inhalation_vapour ATE
    """
    # calculate ATE for each hom material
    inhalation_vapour_ate_dict = {}
    # loop over hom materials
    for hom_material in hom_materials:
        # create a df per hom material
        df_hom_material = df_product.loc[df_product["Homogenous Material"] == hom_material]

        df_calculation = pd.merge(df_hom_material, df_toxicity, on="CAS", how="left")

        # Calculate ATE oral for max_contribution_hom_mat
        LC50 = 'LC50_vapour'
        max_percent_in_hom_mat = "max_contribution_hom_mat"

        #make sure the cols are numeric:
        df_calculation[LC50] = pd.to_numeric(df_calculation[LC50], errors='coerce')
        df_calculation[max_percent_in_hom_mat] = pd.to_numeric(df_calculation[max_percent_in_hom_mat], errors='coerce')

        # drop all values with NaN
        df_calculation = df_calculation.dropna(subset=[LC50, max_percent_in_hom_mat])

        # calcuale in a table form conc / LD50 for each component
        df_calculation["conc_divided_by_LD50"] = df_calculation[max_percent_in_hom_mat]/df_calculation[LC50]

        # sum conc / LD50 of the components
        ate_100 = df_calculation["conc_divided_by_LD50"].sum()

        # calculate the ATE 100/ATE = sum(conc / LD50) -> ATE = 100/sum(conc / LD50)

        ate = 100 / ate_100

        inhalation_vapour_ate_dict[hom_material] = float(ate)

    return inhalation_vapour_ate_dict

def ate_inhalation_dust_mist_aerosol(hom_materials, df_product, df_toxicity):
    """
    :param hom_materials: df with homogenous materials
    :param df_product: df with a product
    :param df_toxicity: df with hazards
    :return: dictionary with inhalation_dust_mist_aerosol ATE
    """
    # calculate ATE for each hom material
    inhalation_dust_mist_aerosol_ate_dict = {}
    # loop over hom materials
    for hom_material in hom_materials:
        # create a df per hom material
        df_hom_material = df_product.loc[df_product["Homogenous Material"] == hom_material]

        df_calculation = pd.merge(df_hom_material, df_toxicity, on="CAS", how="left")

        # Calculate ATE oral for max_contribution_hom_mat
        LC50 = 'LC50_dust_mist_aerosol'
        max_percent_in_hom_mat = "max_contribution_hom_mat"

        #make sure the cols are numeric:
        df_calculation[LC50] = pd.to_numeric(df_calculation[LC50], errors='coerce')
        df_calculation[max_percent_in_hom_mat] = pd.to_numeric(df_calculation[max_percent_in_hom_mat], errors='coerce')

        # drop all values with NaN
        df_calculation = df_calculation.dropna(subset=[LC50, max_percent_in_hom_mat])

        # calcuale in a table form conc / LD50 for each component
        df_calculation["conc_divided_by_LD50"] = df_calculation[max_percent_in_hom_mat]/df_calculation[LC50]

        # sum conc / LD50 of the components
        ate_100 = df_calculation["conc_divided_by_LD50"].sum()

        # calculate the ATE 100/ATE = sum(conc / LD50) -> ATE = 100/sum(conc / LD50)

        ate = 100 / ate_100

        inhalation_dust_mist_aerosol_ate_dict[hom_material] = float(ate)

    return inhalation_dust_mist_aerosol_ate_dict

def ate_dermal(hom_materials, df_product, df_toxicity):
    """
    :param hom_materials: df with homogenous materials
    :param df_product: df with a product
    :param df_toxicity: df with hazards
    :return: dictionary with dermal ATE
    """
    # calculate ATE for each hom material
    inhalation_dermal_ate_dict = {}
    # loop over hom materials
    for hom_material in hom_materials:
        # create a df per hom material
        df_hom_material = df_product.loc[df_product["Homogenous Material"] == hom_material]

        df_calculation = pd.merge(df_hom_material, df_toxicity, on="CAS", how="left")

        # Calculate ATE oral for max_contribution_hom_mat
        LC50 = 'LD50_dermal'
        max_percent_in_hom_mat = "max_contribution_hom_mat"

        #make sure the cols are numeric:
        df_calculation[LC50] = pd.to_numeric(df_calculation[LC50], errors='coerce')
        df_calculation[max_percent_in_hom_mat] = pd.to_numeric(df_calculation[max_percent_in_hom_mat], errors='coerce')

        # drop all values with NaN
        df_calculation = df_calculation.dropna(subset=[LC50, max_percent_in_hom_mat])

        # calcuale in a table form conc / LD50 for each component
        df_calculation["conc_divided_by_LD50"] = df_calculation[max_percent_in_hom_mat]/df_calculation[LC50]

        # sum conc / LD50 of the components
        ate_100 = df_calculation["conc_divided_by_LD50"].sum()

        # calculate the ATE 100/ATE = sum(conc / LD50) -> ATE = 100/sum(conc / LD50)

        ate = 100 / ate_100

        inhalation_dermal_ate_dict[hom_material] = float(ate)

    return inhalation_dermal_ate_dict

ate_oral = ate_oral(hom_materials, df_product, df_toxicity)
ate_inhalation_gas = ate_inhalation_gas(hom_materials, df_product, df_toxicity)
ate_inhalation_vapour = ate_inhalation_vapour(hom_materials, df_product, df_toxicity)
ate_inhalation_dust_mist_aerosol = ate_inhalation_dust_mist_aerosol(hom_materials, df_product, df_toxicity)
ate_dermal = ate_dermal(hom_materials, df_product, df_toxicity)

print(ate_oral, ate_inhalation_gas, ate_inhalation_vapour,ate_inhalation_dust_mist_aerosol, ate_dermal)